# Distribution Plots with Seaborn

Understanding how your data is distributed is a critical first step in any analysis. Seaborn provides a family of functions for visualizing univariate and bivariate distributions — histograms, kernel density estimates, empirical CDFs, and rug plots — all with consistent APIs and built-in support for comparing groups.

This notebook covers:
- `sns.histplot()` — histograms with optional KDE overlay
- `sns.kdeplot()` — kernel density estimation
- `sns.ecdfplot()` — empirical cumulative distribution functions
- `sns.rugplot()` — marginal tick marks
- `sns.displot()` — figure-level distribution plotting with faceting

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
sns.set_theme(style='whitegrid')

In [ ]:
tips = sns.load_dataset('tips')
penguins = sns.load_dataset('penguins').dropna()

print(f'tips shape: {tips.shape}')
print(f'penguins shape: {penguins.shape}')

## Histograms with `sns.histplot()`

A histogram divides the data range into bins and shows the count (or density) of observations in each bin. Seaborn's `histplot()` is the modern replacement for the older `distplot()`.

In [ ]:
# Basic histogram
plt.figure(figsize=(8, 5))
sns.histplot(data=tips, x='total_bill')
plt.title('Distribution of Total Bill')
plt.show()

In [ ]:
# Controlling the number of bins
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, n_bins in zip(axes, [10, 20, 40]):
    sns.histplot(data=tips, x='total_bill', bins=n_bins, ax=ax)
    ax.set_title(f'{n_bins} bins')

plt.tight_layout()
plt.show()

In [ ]:
# Histogram with a KDE curve overlay
plt.figure(figsize=(8, 5))
sns.histplot(data=tips, x='total_bill', kde=True, color='coral')
plt.title('Histogram with KDE Overlay')
plt.show()

### Comparing Groups with `hue` and `multiple`

The `hue` parameter splits the histogram by a categorical variable. The `multiple` parameter controls how overlapping bars are handled:
- `'layer'` (default) — overlapping semi-transparent bars
- `'dodge'` — side-by-side bars
- `'stack'` — stacked bars
- `'fill'` — stacked and normalized to 100%

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, mode in zip(axes.flat, ['layer', 'dodge', 'stack', 'fill']):
    sns.histplot(
        data=tips, x='total_bill', hue='sex',
        multiple=mode, ax=ax, bins=20,
    )
    ax.set_title(f"multiple='{mode}'")

plt.tight_layout()
plt.show()

In [ ]:
# Histogram of penguin bill lengths split by species
plt.figure(figsize=(9, 5))
sns.histplot(
    data=penguins, x='bill_length_mm',
    hue='species', kde=True, bins=25,
)
plt.title('Bill Length Distribution by Species')
plt.show()

## Kernel Density Estimation with `sns.kdeplot()`

KDE smooths the observed data points with a kernel function to produce a continuous density curve. It is especially useful for comparing the shapes of distributions without the bin-width sensitivity of histograms.

In [ ]:
plt.figure(figsize=(8, 5))
sns.kdeplot(data=tips, x='total_bill', fill=True, color='steelblue')
plt.title('KDE of Total Bill')
plt.show()

In [ ]:
# Effect of bandwidth (bw_adjust)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, bw in zip(axes, [0.3, 1.0, 3.0]):
    sns.kdeplot(
        data=tips, x='total_bill',
        bw_adjust=bw, fill=True, ax=ax,
    )
    ax.set_title(f'bw_adjust = {bw}')

plt.tight_layout()
plt.show()

In [ ]:
# Comparing multiple groups
plt.figure(figsize=(9, 5))
sns.kdeplot(
    data=penguins, x='flipper_length_mm',
    hue='species', fill=True, alpha=0.4,
    common_norm=False,
)
plt.title('Flipper Length Density by Species')
plt.show()

In [ ]:
# 2D KDE — bivariate density
plt.figure(figsize=(8, 6))
sns.kdeplot(
    data=penguins, x='bill_length_mm', y='bill_depth_mm',
    hue='species', fill=True, alpha=0.5,
    levels=5,
)
plt.title('Bivariate KDE: Bill Length vs Depth')
plt.show()

## Empirical CDF with `sns.ecdfplot()`

The empirical cumulative distribution function shows, for each value, the proportion of observations that fall at or below it. ECDFs avoid the binning decisions of histograms and display every data point.

In [ ]:
plt.figure(figsize=(8, 5))
sns.ecdfplot(data=tips, x='total_bill')
plt.title('Empirical CDF of Total Bill')
plt.ylabel('Proportion')
plt.show()

In [ ]:
# ECDF comparison across species
plt.figure(figsize=(8, 5))
sns.ecdfplot(data=penguins, x='body_mass_g', hue='species')
plt.title('Body Mass ECDF by Species')
plt.show()

## Rug Plots with `sns.rugplot()`

A rug plot draws thin vertical lines at each data point along an axis. It is most useful as a complement to KDE or histogram plots to show the actual data density.

In [ ]:
plt.figure(figsize=(8, 5))
sns.kdeplot(data=tips, x='total_bill', fill=True, alpha=0.3)
sns.rugplot(data=tips, x='total_bill', height=0.05, color='black', alpha=0.5)
plt.title('KDE with Rug Plot')
plt.show()

In [ ]:
# Rug plot with hue
plt.figure(figsize=(8, 5))
sns.kdeplot(data=tips, x='tip', hue='time', fill=True, alpha=0.4)
sns.rugplot(data=tips, x='tip', hue='time', height=0.06, alpha=0.6)
plt.title('Tip Distribution with Rug Marks — Lunch vs Dinner')
plt.show()

## Figure-level Distributions: `sns.displot()`

`displot()` is the **figure-level** interface that unifies histograms, KDEs, and ECDFs. It returns a `FacetGrid`, so you can facet across rows and columns.

Use `kind` to choose the plot type:
- `'hist'` (default)
- `'kde'`
- `'ecdf'`

In [ ]:
# Faceted histogram by day of week
g = sns.displot(
    data=tips, x='total_bill',
    col='day', col_wrap=2,
    kind='hist', kde=True,
    height=3.5, aspect=1.2,
)
g.fig.suptitle('Total Bill Distribution by Day', y=1.03)
plt.show()

In [ ]:
# Faceted KDE
g = sns.displot(
    data=penguins, x='bill_length_mm',
    col='species', hue='sex',
    kind='kde', fill=True,
    height=4, aspect=1,
)
g.fig.suptitle('Bill Length KDE by Species and Sex', y=1.03)
plt.show()

In [ ]:
# Faceted ECDF
g = sns.displot(
    data=penguins, x='body_mass_g',
    col='island',
    kind='ecdf', hue='species',
    height=4, aspect=1.1,
)
plt.show()

## Combining Histogram, KDE, and Rug in One Plot

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(
    data=tips, x='tip',
    bins=20, kde=True, stat='density',
    color='salmon', alpha=0.5,
)
sns.rugplot(data=tips, x='tip', color='darkred', alpha=0.4, height=0.04)
plt.title('Tip Distribution: Histogram + KDE + Rug')
plt.show()

## Key Takeaways

| Function | Level | Description |
|---|---|---|
| `histplot()` | Axes | Histogram with optional KDE overlay |
| `kdeplot()` | Axes | Smooth kernel density estimate (1D or 2D) |
| `ecdfplot()` | Axes | Step function showing cumulative proportion |
| `rugplot()` | Axes | Tick marks at each observation |
| `displot()` | Figure | Unified distribution plotter with faceting |

- Use `kde=True` on `histplot()` to get the best of both worlds.
- `bw_adjust` on `kdeplot()` controls smoothing: lower values show more detail, higher values smooth more.
- `displot()` is the go-to for quick faceted distribution comparisons.
- Always consider ECDFs as an alternative to histograms — they show every data point and require no binning decisions.